# 03 - Arrest Prediction (Classification)

This notebook trains and compares the three supervised models called out in
the proposal:

1. Logistic Regression (interpretable linear baseline)
2. Random Forest (non-linear ensemble)
3. SVM with RBF kernel (non-linear margin-based)

It also performs 5-fold cross-validation, looks at the RF calibration curve,
and discusses the class-imbalance trade-offs.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.clean import clean, load_raw
from src.data.features import engineer_features, get_feature_matrix
from src.models.classification import run_cross_validation_report, train_evaluate_all

sns.set_theme(style="whitegrid")

df_raw = load_raw(str(repo_root / "data" / "raw" / "chicago_crimes.csv"))
df_clean = clean(df_raw)
df_feat = engineer_features(df_clean, persist_encoders=False)
X, y, feature_cols = get_feature_matrix(df_feat)

print("X shape:", X.shape)
print("Arrest rate:", y.mean())

## 1. Train all three models with class-imbalance handling

`train_evaluate_all` applies:

- `StandardScaler` on features
- Stratified 80/20 train/test split
- SMOTE oversampling on the training fold
- `RandomizedSearchCV` hyperparameter search for RF (and LR when `tune_lr=True`)
- `CalibratedClassifierCV` on RF to improve probability estimates

In [ ]:
out = train_evaluate_all(
    X,
    y,
    tune_rf=True,
    tune_lr=True,
    tune_svm=False,
    calibrate_rf=True,
    use_smote=True,
)

results = out["results"]
models = out["models"]
X_test = out["X_test"]
y_test = out["y_test"]

results_df = pd.DataFrame(
    {k: {kk: vv for kk, vv in v.items() if kk != "model"} for k, v in results.items()}
).T[["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "Train Time (s)"]]
results_df

## 2. Confusion matrices and ROC curves

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, auc, roc_curve

plottable = {k: v for k, v in models.items() if hasattr(v, "predict")}
n = len(plottable)
fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 4))
if n == 1:
    axes = [axes]
for ax, (name, model) in zip(axes, plottable.items()):
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test, ax=ax, display_labels=["No Arrest", "Arrest"]
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()

fig = plt.figure(figsize=(8, 6))
for name, model in plottable.items():
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc(fpr, tpr):.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves - test set")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 3. Random Forest feature importance

In [ ]:
rf = models.get("Random Forest")
if rf is not None and hasattr(rf, "feature_importances_"):
    importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(x=importances.values, y=importances.index, ax=ax, orient="h", color="#1f77b4")
    ax.set_title("Random Forest feature importances")
    plt.tight_layout()
    plt.show()
    display(importances.to_frame("importance"))

## 4. Calibration curve - does RF output reliable probabilities?

A perfect model traces the diagonal. SMOTE tends to push raw RF probabilities
higher than the true rate, which is why we add `CalibratedClassifierCV` with
sigmoid calibration and publish it as "Random Forest (Calibrated)".

In [ ]:
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(7, 6))
for name in ("Random Forest", "Random Forest (Calibrated)", "Logistic Regression"):
    model = models.get(name)
    if model is None or not hasattr(model, "predict_proba"):
        continue
    y_prob = model.predict_proba(X_test)[:, 1]
    prob_true, prob_pred = calibration_curve(y_test, y_prob, n_bins=10, strategy="quantile")
    ax.plot(prob_pred, prob_true, marker="o", label=name)

ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Perfect calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed positive rate")
ax.set_title("Reliability (calibration) curve")
ax.legend()
plt.tight_layout()
plt.show()

## 5. 5-fold cross-validation

Confirms the single test-set numbers are not a lucky split.

In [ ]:
cv_df = run_cross_validation_report(
    X, y, cv=5, report_path=str(repo_root / "outputs" / "reports" / "cv_results.csv")
)
cv_df

## Classification Takeaways

- **Random Forest is the strongest model** by ROC-AUC and F1. Its non-linear
  handling of `PrimaryType_enc`, `LocationDesc_enc`, and geographic features
  is what drives the gap over Logistic Regression.
- **Logistic Regression remains a useful baseline** - interpretable coefficients
  let us verify that the tree model is behaving reasonably on the same signals.
- **SVM-RBF sits between the two** and is roughly an order of magnitude slower
  to train; we cap its training size to keep the pipeline practical.
- **Class imbalance is handled by stacking three techniques**: `class_weight="balanced"`,
  SMOTE on the training fold, and threshold-aware ROC/PR metrics rather than raw accuracy.
- **RF calibration matters** - raw RF probabilities are pushed up by SMOTE,
  and `CalibratedClassifierCV` brings them back toward the diagonal on the
  reliability curve.